# 04 · Prompt engineering

**Goal:** improve a weak prompt one technique at a time, and **measure** each change with the eval from notebook 03, so you know what actually helped.

| | |
|---|---|
| **You will learn** | Being clear and direct, being specific, structuring with XML tags, and giving examples |
| **Prerequisites** | [Notebook 03](03_prompt_evaluation.ipynb) |
| **Run time** | About 4 minutes |
| **Cost** | About $0.29 when I ran it. This notebook stops itself at $0.40 |
| **Paid cells** | Sections 2 to 6. Section 1 is free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## The method

Prompt engineering is a loop, not a single clever rewrite:

```
 1. Set a goal ──▶ 2. Write a first prompt ──▶ 3. Evaluate it
                                                    │
               5. Evaluate again ◀── 4. Apply ONE technique
                     │
                     └──▶ keep what helps, then repeat 4 and 5
```

**Change one thing at a time.** If you change three things and the score moves, you won't know which one did it.

**Our task, from the course:** a one-day meal plan for an athlete, based on their height, weight, goal and dietary restrictions. We start with a deliberately weak prompt and improve it in sections 2 to 5. Section 6 tests the last technique, examples, on a second, smaller task where the effect is easier to see.

## Setup

The same setup as the earlier notebooks. Run the cells from top to bottom.

In [1]:
import os
from statistics import mean

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [2]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.40     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def ask(prompt, max_tokens=2000):
    """Send one prompt and return the text of the answer."""
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    ))
    return text_from_message(response)

## 1. The test set and the grader (free)

**The dataset.** Notebook 03 showed how to have Claude generate test cases. Here we write three by hand instead. It's free, and each athlete is chosen to test something different.

**The grader.** This is the model grader from notebook 03, with one addition: `criteria` describes what a good meal plan must contain, so the grader judges every version by the same standard. Defining it now, before writing any prompt, is part of the method.

In [3]:
athletes = [
    {"height": "182 cm", "weight": "84 kg", "goal": "Build muscle", "restrictions": "Vegetarian"},
    {"height": "165 cm", "weight": "58 kg", "goal": "Marathon training, improve endurance", "restrictions": "Gluten-free, on a tight budget"},
    {"height": "175 cm", "weight": "95 kg", "goal": "Lose fat while keeping strength", "restrictions": "No dairy, no pork"},
]

CRITERIA = """
A good meal plan must include:
- A daily calorie total that suits the athlete's size and goal
- A breakdown of protein, fat and carbohydrates
- Every meal with exact foods, portion sizes, and the time to eat it
- Only foods that respect every dietary restriction
"""


class Grade(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int = Field(ge=1, le=10)


def grade_by_model(athlete, output):
    prompt = f"""
You are an expert sports nutritionist. Evaluate this AI-generated meal plan.

<athlete>
{athlete}
</athlete>

<meal_plan>
{output}
</meal_plan>

<criteria>
{CRITERIA}
</criteria>

Judge the meal plan against the criteria. Give 1-3 key strengths, 1-3 key weaknesses,
a concise reasoning, and a score from 1 to 10.
"""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Grade,
    ))
    return response.parsed_output


def run_eval(build_prompt, label):
    """Run a prompt version on every athlete, grade each answer, and return the scores."""
    print(f"=== {label}")
    results = []
    for athlete in athletes:
        output = ask(build_prompt(athlete))
        grade = grade_by_model(athlete, output)
        results.append({"athlete": athlete, "output": output, "score": grade.score, "grade": grade})

    print(f"\n{'score':>6}  athlete")
    for r in results:
        print(f"{r['score']:>6}  {r['athlete']['goal']} ({r['athlete']['restrictions']})")
    print(f"\nAverage: {mean(r['score'] for r in results):.2f} / 10")
    return results


def show_feedback(results):
    """Print the grader's view of the lowest-scoring answer, the one to learn most from."""
    worst = min(results, key=lambda r: r["score"])
    print(f"Lowest score: {worst['score']} for {worst['athlete']['goal']}")
    print("Weaknesses :", worst["grade"].weaknesses)


scoreboard = {}
print(f"{len(athletes)} athletes, and the grader is ready.")

3 athletes, and the grader is ready.


## 2. The starting point (paid)

A deliberately weak first prompt: a vague question, with the details tacked on below. Don't worry about a low score. The point is to have a **baseline** to improve on.

In [6]:
def prompt_v1(athlete):
    return f"""What should this person eat?

- Height: {athlete['height']}
- Weight: {athlete['weight']}
- Goal: {athlete['goal']}
- Dietary restrictions: {athlete['restrictions']}"""


v1 = run_eval(prompt_v1, "v1: a vague question")
scoreboard["v1: vague question"] = mean(r["score"] for r in v1)
show_feedback(v1)

=== v1: a vague question
[input=64 output=1,053 cost=$0.01066 | spent $0.0779 of $0.40]
[input=1,714 output=485 cost=$0.00828 | spent $0.0862 of $0.40]
[input=77 output=1,289 cost=$0.01304 | spent $0.0993 of $0.40]
[input=1,963 output=473 cost=$0.00866 | spent $0.1079 of $0.40]
[input=72 output=1,155 cost=$0.01169 | spent $0.1196 of $0.40]
[input=1,824 output=503 cost=$0.00868 | spent $0.1283 of $0.40]

 score  athlete
     5  Build muscle (Vegetarian)
     5  Marathon training, improve endurance (Gluten-free, on a tight budget)
     6  Lose fat while keeping strength (No dairy, no pork)

Average: 5.33 / 10
Lowest score: 5 for Build muscle
Weaknesses : ["Meals lack exact portions and times of day: many items are unquantified (e.g., 'cheese', 'vegetables', 'rice', 'curry') and no meal timing is given", "Sample day numbers don't add up: meal calories total ~3,000 but protein is ~180 g, above the stated 140-170 g range, and macros are not given per meal for carbs and fat", "Offers options

## 3. Technique 1: be clear and direct (paid)

The first line is the most important part of a prompt. Make it:

- **Clear:** say exactly what you want, in plain language.
- **Direct:** give an instruction, not a question, and start with an action verb such as *Write*, *Generate* or *List*.

| Instead of | Write |
|---|---|
| "I need to know about those solar panel things on roofs" | "Write three paragraphs about how solar panels work." |
| "What countries use geothermal energy?" | "Identify three countries that use geothermal energy. Include generation stats for each." |

**The only change from v1 is the first line.**

In [7]:
def prompt_v2(athlete):
    return f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {athlete['height']}
- Weight: {athlete['weight']}
- Goal: {athlete['goal']}
- Dietary restrictions: {athlete['restrictions']}"""


v2 = run_eval(prompt_v2, "v2: clear and direct")
scoreboard["v2: clear and direct"] = mean(r["score"] for r in v2)
show_feedback(v2)

=== v2: clear and direct
[input=85 output=1,288 cost=$0.01305 | spent $0.1413 of $0.40]
[input=1,949 output=516 cost=$0.00906 | spent $0.1504 of $0.40]
[input=98 output=1,373 cost=$0.01393 | spent $0.1643 of $0.40]
[input=2,047 output=497 cost=$0.00906 | spent $0.1734 of $0.40]
[input=93 output=1,240 cost=$0.01259 | spent $0.1860 of $0.40]
[input=1,909 output=503 cost=$0.00885 | spent $0.1948 of $0.40]

 score  athlete
     7  Build muscle (Vegetarian)
     6  Marathon training, improve endurance (Gluten-free, on a tight budget)
     6  Lose fat while keeping strength (No dairy, no pork)

Average: 6.33 / 10
Lowest score: 6 for Marathon training, improve endurance
Weaknesses : ["Math is inconsistent: the meal macros sum to about 460 g carbs and ~127 g protein, but the stated targets are 350-450 g carbs and 85-105 g protein, so the plan exceeds its own ranges; the per-meal calories also don't add up to the total, and the snack fat values are missing", 'Meal times are mostly absent; only 

## 4. Technique 2: be specific (paid)

A clear task can still be done a hundred different ways. Specific guidelines tell Claude which way you want. There are two kinds:

| Kind | What it controls | When to use it |
|---|---|---|
| **Output guidelines** | What the answer must contain: length, structure, required parts | Almost always |
| **Process steps** | How Claude should work through the problem | Complex tasks such as troubleshooting or decisions, where you want several angles considered |

Our grader's criteria tell us exactly what's missing, so we turn them into output guidelines. **The only change from v2 is the guidelines list.**

In [8]:
def prompt_v3(athlete):
    return f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {athlete['height']}
- Weight: {athlete['weight']}
- Goal: {athlete['goal']}
- Dietary restrictions: {athlete['restrictions']}

Guidelines:
1. Include an accurate daily calorie total
2. Show the protein, fat and carbohydrate amounts
3. Say when to eat each meal
4. Use only foods that fit the restrictions
5. List every portion size in grams
6. Keep it budget-friendly if the athlete mentions a budget"""


v3 = run_eval(prompt_v3, "v3: specific guidelines")
scoreboard["v3: specific guidelines"] = mean(r["score"] for r in v3)
show_feedback(v3)

=== v3: specific guidelines
[input=191 output=1,306 cost=$0.01344 | spent $0.2083 of $0.40]
[input=1,967 output=646 cost=$0.01039 | spent $0.2187 of $0.40]
[input=204 output=2,000 cost=$0.02041 | spent $0.2391 of $0.40]
[input=1,087 output=513 cost=$0.00730 | spent $0.2464 of $0.40]
[input=199 output=1,416 cost=$0.01456 | spent $0.2609 of $0.40]
[input=2,085 output=539 cost=$0.00956 | spent $0.2705 of $0.40]

 score  athlete
     6  Build muscle (Vegetarian)
     3  Marathon training, improve endurance (Gluten-free, on a tight budget)
     7  Lose fat while keeping strength (No dairy, no pork)

Average: 5.33 / 10
Lowest score: 3 for Marathon training, improve endurance
Weaknesses : ['The plan is truncated: only breakfast is given, so there is no lunch, dinner, snacks or pre/during/post-run fueling, and the stated 2,600 kcal is never reached (breakfast is about 764 kcal)', 'No daily totals are shown, so the macro breakdown cannot be checked against the targets; the stated targets also d

**What to notice**

- **When I ran this, v3 scored lower than v2 (5.33 against 6.33), but not because of the guidelines.** The marathon plan used exactly 2,000 output tokens, our `max_tokens`, and was cut off after breakfast. The grader rightly gave that plan 3/10. The other two athletes scored 6 and 7.
- **This is the `stop_reason` lesson from notebook 00, inside an eval.** `ask()` never checks it, so a cut-off answer looks like a bad prompt. A real eval should record `stop_reason` for every answer, and you can add that in Your turn.
- Telling Claude exactly what a good answer contains is often the biggest single improvement, as long as the answer has room to finish.
- The guidelines came straight from the grader's criteria. **Your eval tells you what to write in your prompt.**

## 5. Technique 3: structure with XML tags (paid)

When a prompt mixes instructions with data, Claude has to work out which text is which. XML tags draw clear boundaries around each piece:

```
<athlete_information>
- Height: 182 cm
...
</athlete_information>
```

- **Pick descriptive names.** `<athlete_information>` says more than `<data>`, and `<my_code>` and `<docs>` say more than one unlabeled block.
- **They matter most for big prompts** that mix long documents, code and data. You've already seen them in the graders, where `<task>` and `<solution>` keep the two apart.

**The only change from v3 is the tags**, plus a line pointing at them.

In [9]:
def prompt_v4(athlete):
    return f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information>
- Height: {athlete['height']}
- Weight: {athlete['weight']}
- Goal: {athlete['goal']}
- Dietary restrictions: {athlete['restrictions']}
</athlete_information>

Base the plan on the athlete information above.

<guidelines>
1. Include an accurate daily calorie total
2. Show the protein, fat and carbohydrate amounts
3. Say when to eat each meal
4. Use only foods that fit the restrictions
5. List every portion size in grams
6. Keep it budget-friendly if the athlete mentions a budget
</guidelines>"""


v4 = run_eval(prompt_v4, "v4: XML tags")
scoreboard["v4: XML tags"] = mean(r["score"] for r in v4)
show_feedback(v4)

=== v4: XML tags
[input=228 output=1,861 cost=$0.01907 | spent $0.2896 of $0.40]
[input=2,522 output=521 cost=$0.01025 | spent $0.2998 of $0.40]
[input=241 output=1,537 cost=$0.01585 | spent $0.3157 of $0.40]
[input=2,211 output=672 cost=$0.01114 | spent $0.3268 of $0.40]
[input=236 output=1,274 cost=$0.01321 | spent $0.3400 of $0.40]
[input=1,943 output=488 cost=$0.00877 | spent $0.3488 of $0.40]

 score  athlete
     7  Build muscle (Vegetarian)
     7  Marathon training, improve endurance (Gluten-free, on a tight budget)
     7  Lose fat while keeping strength (No dairy, no pork)

Average: 7.00 / 10
Lowest score: 7 for Build muscle
Weaknesses : ["The plan's stated target (about 3,000 kcal, 165 g protein) does not match the actual plan (about 3,387 kcal, 200 g protein); it is internally inconsistent and the user must make manual edits to hit the target.", "The note's claim of a 'roughly +600-900 kcal' surplus conflicts with the earlier 300-400 kcal surplus framing, and the totals/adj

**What to notice**

- **When I ran this, v4 scored 7.00 for all three athletes**, the best so far, and no answer was cut off. Part of the gain over v3 is simply that nothing was truncated, so compare v4 with v2 (6.33) as well.
- Don't read too much into the tags alone. This prompt is short, and its parts were already easy to tell apart. Tags earn their place as prompts grow, and they cost almost nothing to add.
- **Read the weaknesses in every version:** the grader keeps finding numbers that don't add up, such as calorie totals that don't match the meals. A guideline asking for accuracy didn't fix that. Arithmetic is better done by code, as notebook 06 shows with code execution.
- **If the score dropped a little, that may be noise**, not the tags. Three test cases and a model grader don't give precise numbers, as notebook 03 explained.

### The scoreboard so far

In [ ]:
print(f"{'version':<26}{'average':>8}")
for name, score in scoreboard.items():
    print(f"{name:<26}{score:>8.2f}")

print("\nThe v4 answer for the first athlete, up to 1,200 characters:\n")
print(v4[0]["output"][:1200])

## 6. Technique 4: give examples (paid)

Some things are easier to **show** than to describe. Giving one example is called **one-shot** prompting, and giving several is **multi-shot**. Examples are especially good for:

- tricky corner cases,
- an exact output format,
- a particular tone or style.

Meal plans are long, so examples there would be expensive to test. Instead we use the course's second task: **sentiment classification**, where the tricky case is **sarcasm**. Answers are a single word, so a **free code grader** can check them, and we can compare zero examples against several.

Each answer must be exactly `positive` or `negative`, because a program will read it. The labels below are ours, written for this test.

**The two prompts are identical except for the examples**, so any difference in score comes from them.

In [5]:
tweets = [
    ("Great game tonight, what a comeback!", "positive"),
    ("Oh fantastic, my train is cancelled again. Love that for me.", "negative"),
    ("Wow, another software update that deletes my settings. Brilliant work.", "negative"),
    ("This coffee shop never disappoints. Best flat white in town.", "positive"),
    ("Sure, waiting 40 minutes on hold was exactly how I wanted to spend lunch.", "negative"),
    ("Finally finished my first 10k run. Legs hurt but I'm so proud!", "positive"),
    ("Yeah, the best movie since Plan 9 from Outer Space. Truly unforgettable.", "negative"),
    ("Can't believe how fast the delivery was, arrived the same day!", "positive"),
]

ZERO_SHOT = """Classify the sentiment of this tweet as positive or negative.
Reply with exactly one lowercase word: positive or negative.

<tweet>
{tweet}
</tweet>"""

MULTI_SHOT = """Classify the sentiment of this tweet as positive or negative.
Reply with exactly one lowercase word: positive or negative.

Here are example tweets with ideal answers:

<example>
<sample_input>Loved every minute of the concert last night!</sample_input>
<ideal_output>positive</ideal_output>
</example>

<example>
<sample_input>Oh great, a flight delay tonight. Just what I needed. Excellent!</sample_input>
<ideal_output>negative</ideal_output>
</example>

The second example is negative because the praise is sarcastic: nobody is pleased about a delay.

<tweet>
{tweet}
</tweet>"""


def classify_all(template, label):
    print(f"=== {label}")
    correct = 0
    for tweet, expected in tweets:
        answer = ask(template.format(tweet=tweet), max_tokens=300).strip()
        is_correct = answer == expected  # code grader: exact match, so the format counts too
        correct += is_correct
        print(f"   {'ok ' if is_correct else 'BAD'}  expected {expected:<8}  got {answer[:40]!r}")
    accuracy = correct / len(tweets)
    print(f"\nScore: {correct}/{len(tweets)} ({accuracy:.0%})\n")
    return accuracy


zero_shot_accuracy = classify_all(ZERO_SHOT, "zero-shot: no examples")
multi_shot_accuracy = classify_all(MULTI_SHOT, "multi-shot: two examples")

assert 0 <= zero_shot_accuracy <= 1 and 0 <= multi_shot_accuracy <= 1

=== zero-shot: no examples
[input=75 output=4 cost=$0.00019 | spent $0.0615 of $0.40]
   ok   expected positive  got 'positive'
[input=85 output=4 cost=$0.00021 | spent $0.0617 of $0.40]
   ok   expected negative  got 'negative'
[input=84 output=4 cost=$0.00021 | spent $0.0619 of $0.40]
   ok   expected negative  got 'negative'
[input=82 output=4 cost=$0.00020 | spent $0.0622 of $0.40]
   ok   expected positive  got 'positive'
[input=94 output=4 cost=$0.00023 | spent $0.0624 of $0.40]
   ok   expected negative  got 'negative'
[input=88 output=4 cost=$0.00022 | spent $0.0626 of $0.40]
   ok   expected positive  got 'positive'
[input=94 output=4 cost=$0.00023 | spent $0.0628 of $0.40]
   ok   expected negative  got 'negative'
[input=83 output=4 cost=$0.00021 | spent $0.0630 of $0.40]
   ok   expected positive  got 'positive'

Score: 8/8 (100%)

=== multi-shot: two examples
[input=235 output=4 cost=$0.00051 | spent $0.0635 of $0.40]
   ok   expected positive  got 'positive'
[input=245 out

**What to notice**

- Look at the `BAD` rows. A wrong label is usually missed sarcasm. An answer in the wrong format, such as `Negative.`, also counts as wrong, because a program reading it would break.
- **Our examples are different tweets from the ones being tested.** If you put a test case in the prompt as an example, the score stops meaning anything, because Claude would be copying the answer.
- We also explained **why** the second example is negative. An explanation helps Claude apply the idea to new cases, not just copy a pattern.
- **When I ran this, both versions scored 8/8.** Sonnet 5.5 caught every sarcastic tweet with no examples at all. That's a useful result too: examples matter most where your eval shows failures, and here there were none to fix. Harder cases, or a format that's hard to describe in words, are where examples pay off.

**Finding good examples:** run an eval, take your highest-scoring outputs, and use them as examples in the next version of the prompt. Try it with the meal plan in Your turn.

## Final spend

In [11]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.3488 of its $0.40 budget.


## Takeaways

1. **Improve prompts in a loop:** evaluate, apply one technique, evaluate again.
2. **Be clear and direct:** start with an instruction and an action verb, not a question.
3. **Be specific:** list what the answer must contain, and add process steps for complex reasoning. Your grader's criteria are a good source of guidelines.
4. **Use XML tags** to separate instructions from data. They matter more as prompts grow.
5. **Give examples** for corner cases and exact formats. Wrap them in tags, explain why they're good, and never reuse your test cases as examples.
6. **Record `stop_reason` in every eval.** A cut-off answer looks like a bad prompt, as v3 showed.
7. **Small evals are noisy.** Trust big jumps, and re-run before believing small ones.

## Your turn

The first exercise costs about 6 cents. The others cost a cent or two.

- **Catch truncation:** make `ask()` print a warning when `response.stop_reason == "max_tokens"`, raise `max_tokens` to 3000, and re-run v3. How does it score with room to finish? (About 8 cents.)
- **One-shot meal plan:** take the highest-scoring meal plan from `v4`, put it in `prompt_v5` inside `<ideal_output>` tags with a line on why it's good, and run `run_eval`. Use a different athlete as the example, so you're not grading Claude on its own answer.
- Add **process steps** to `prompt_v3`, such as "first work out the daily calories, then split them across meals". Does the score change?
- In section 6, delete the sentence explaining why the second example is negative. Does the score change without the explanation?
- Add two more sarcastic tweets of your own to `tweets`. Do both versions still score the same?

**Next:** [05 · Tool use](05_tool_use.ipynb): giving Claude functions it can call to get live data and take actions.